# Tests statistiques — biais de domaine et rééquilibrage (`report.tex`)

Ce notebook répond aux deux `\todo[inline]` de `reports/report.tex` portant sur des tests
statistiques manquants :

1. **§ Transformation et data augmentation** (ligne ~496) : un test à un échantillon sur
   `grid_search_dataset_full_1` pour appuyer formellement « le CNN retrouve le site quasi
   parfaitement quel que soit le prétraitement ».
2. **§ Grid search sur le rééquilibrage** (ligne ~763) : un test apparié (t apparié / Wilcoxon
   signé) sur `val_f1_macro` / `val_min_critical_recall` entre configurations de
   `grid_search_imbalance_3` (tâche `type`), et un état des lieux côté `criticality` où
   l'appariement n'est pas encore possible.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


In [2]:
leaderboard_path = Path.cwd().parent / "results" / "leaderboard.csv"
df = pd.read_csv(leaderboard_path, low_memory=False, parse_dates=["timestamp"])
df.shape


(1345, 85)

## 1. Biais de domaine — `grid_search_dataset_full_1` contre le hasard (0,5)

Le grid balaie 200 configurations de prétraitement (tâche annexe : prédire le site
d'acquisition, Barcelone contre Munich) avec un pas d'apprentissage réduit ($10^{-3}$), sans
collapse. Le leaderboard ne contient **pas** de `test_acc` pour ce grid : ces runs de
diagnostic n'évaluent qu'un split de validation (`metrics_test.json` vide), la précision
pertinente est donc **`val_acc`** — c'est bien la colonne dont les statistiques (min 0,87,
médiane 0,99) correspondent aux chiffres cités dans le rapport.

In [3]:
grid1 = df[(df["name"] == "grid_search_dataset_full_1") & (df["status"] == "ok")].copy()
assert len(grid1) == 200, f"attendu 200 configurations, trouvé {len(grid1)}"
assert grid1["test_acc"].notna().sum() == 0, "test_acc inattendu : vérifier la colonne utilisée"

acc = grid1["val_acc"].dropna()
print(f"n = {len(acc)}")
acc.describe()


n = 200


count    200.000000
mean       0.991745
std        0.012272
min        0.870588
25%        0.992157
50%        0.992157
75%        0.997059
max        1.000000
Name: val_acc, dtype: float64

### Test à un échantillon contre le niveau de hasard (0,5)

$H_0$ : la précision moyenne du CNN à retrouver le site vaut 0,5 (hasard, harmonisation
parfaite). $H_1$ (unilatérale) : elle est supérieure à 0,5 (biais de domaine résiduel).

La distribution est très resserrée près de 1 (plafond), donc clairement non gaussienne :
on complète le t-test (paramétrique) par un test des rangs signés de Wilcoxon contre 0,5
(non paramétrique), qui ne suppose pas la normalité.

In [4]:
chance_level = 0.5

t_stat, p_t = stats.ttest_1samp(acc, popmean=chance_level, alternative="greater")
w_stat, p_w = stats.wilcoxon(acc - chance_level, alternative="greater")

# Taille d'effet (Cohen's d, échantillon unique)
cohens_d = (acc.mean() - chance_level) / acc.std(ddof=1)

# Normalité (informatif : justifie le recours à Wilcoxon en complément du t-test)
shapiro_stat, shapiro_p = stats.shapiro(acc)

print(f"n = {len(acc)}, moyenne = {acc.mean():.4f}, écart-type = {acc.std(ddof=1):.4f}")
print(f"médiane = {acc.median():.4f}, min = {acc.min():.4f}, max = {acc.max():.4f}")
print()
print(f"Shapiro-Wilk (normalité)   : W = {shapiro_stat:.3f}, p = {shapiro_p:.2e}")
print(f"t-test à un échantillon    : t({len(acc) - 1}) = {t_stat:.2f}, p = {p_t:.2e}")
print(f"Wilcoxon signé (contre 0,5): W = {w_stat:.1f}, p = {p_w:.2e}")
print(f"Cohen's d                  : {cohens_d:.2f}")


n = 200, moyenne = 0.9917, écart-type = 0.0123
médiane = 0.9922, min = 0.8706, max = 1.0000

Shapiro-Wilk (normalité)   : W = 0.518, p = 3.85e-23
t-test à un échantillon    : t(199) = 566.68, p = 0.00e+00
Wilcoxon signé (contre 0,5): W = 20100.0, p = 2.18e-35
Cohen's d                  : 40.07


**Conclusion.** Les deux tests rejettent très largement $H_0$ (p négligeable, taille
d'effet énorme) : la précision du CNN à identifier le site est statistiquement — et pas
seulement visuellement — très supérieure au hasard, quelle que soit la configuration de
prétraitement testée. Cela appuie formellement le constat du rapport : le biais de domaine
« quasi parfaitement » détecté n'est pas un artefact d'échantillonnage, il est présent dans
la quasi-totalité des 200 configurations.

## 2. Rééquilibrage — comparaisons appariées sur `grid_search_imbalance_3` (tâche `type`)

Le grid comporte 47 configurations (croisement de `loss.beta`, `loss.gamma`,
`loss.lambda_cost` / `loss.cost_matrix`, `train.sampler_alpha`), chacune rejouée sur les
**mêmes 3 seeds** (0, 1, 2) — un plan en mesures répétées : pour chaque seed, le split
train/val et l'initialisation sont partagés entre configurations, ce qui justifie un test
**apparié** (par seed) plutôt qu'un test sur échantillons indépendants.

On isole les deux configurations comparées dans le tableau `tab:imbalance-grid` du
rapport : la meilleure combinaison *sans* coût clinique (« class-balanced + focale +
sampler ») et la configuration *retenue* (coût calibré + sampler plein + `beta=0,999`).

In [ ]:
grid3 = df[(df["name"] == "grid_search_imbalance_3") & (df["status"] == "ok")].copy()

cfg_cols = [
    "loss.beta",
    "loss.gamma",
    "loss.lambda_cost",
    "loss.cost_matrix",
    "train.sampler",
    "train.sampler_alpha",
]
n_configs = grid3.drop_duplicates(subset=cfg_cols).shape[0]
seed_counts = grid3.groupby(cfg_cols, dropna=False)["seed"].nunique()

print(f"{len(grid3)} runs, {n_configs} configurations distinctes")
print(
    f"configurations avec exactement 3 seeds distincts : {(seed_counts == 3).sum()} / {n_configs}"
)


141 runs, 47 configurations distinctes
configurations avec exactement 3 seeds distincts : 47 / 47


In [ ]:
def select_config(data, **params):
    """Sélectionne les runs d'une configuration exacte (colonnes de cfg_cols)."""
    mask = pd.Series(True, index=data.index)
    for col, value in params.items():
        if value is None or (isinstance(value, float) and np.isnan(value)):
            mask &= data[col].isna()
        else:
            mask &= data[col] == value
    return data[mask]


sans_cout = select_config(
    grid3,
    **{
        "loss.beta": 0.9999,
        "loss.gamma": 1.0,
        "loss.lambda_cost": 0.0,
        "loss.cost_matrix": np.nan,
        "train.sampler": "weight",
        "train.sampler_alpha": 1.0,
    },
)
retenue = select_config(
    grid3,
    **{
        "loss.beta": 0.999,
        "loss.gamma": 0.0,
        "loss.lambda_cost": 1.0,
        "loss.cost_matrix": "[[0.0, 0.5, 1.0], [2.0, 0.0, 1.0], [4.0, 3.0, 0.0]]",
        "train.sampler": "weight",
        "train.sampler_alpha": 1.0,
    },
)

print(
    "sans coût :", sans_cout[["seed", "val_f1_macro", "val_min_critical_recall"]].to_dict("records")
)
print(
    "retenue   :", retenue[["seed", "val_f1_macro", "val_min_critical_recall"]].to_dict("records")
)


sans coût : [{'seed': 0, 'val_f1_macro': 0.8507784008979797, 'val_min_critical_recall': 0.8636363744735718}, {'seed': 1, 'val_f1_macro': 0.8786952495574951, 'val_min_critical_recall': 0.8867924809455872}, {'seed': 2, 'val_f1_macro': 0.8726022839546204, 'val_min_critical_recall': 0.7840909361839294}]
retenue   : [{'seed': 0, 'val_f1_macro': 0.8742985129356384, 'val_min_critical_recall': 0.9090909361839294}, {'seed': 1, 'val_f1_macro': 0.8631390929222107, 'val_min_critical_recall': 0.9339622855186462}, {'seed': 2, 'val_f1_macro': 0.8760389089584351, 'val_min_critical_recall': 0.8863636255264282}]


In [ ]:
paired = (
    sans_cout[["seed", "val_f1_macro", "val_min_critical_recall"]]
    .merge(
        retenue[["seed", "val_f1_macro", "val_min_critical_recall"]],
        on="seed",
        suffixes=("_sans_cout", "_retenue"),
    )
    .sort_values("seed")
)
assert len(paired) == 3, f"attendu 3 seeds appariés (0/1/2), trouvé {len(paired)}"
paired


,seed,val_f1_macro_sans_cout,val_min_critical_recall_sans_cout,val_f1_macro_retenue,val_min_critical_recall_retenue
0,0,0.850778,0.863636,0.874299,0.909091
1,1,0.878695,0.886792,0.863139,0.933962
2,2,0.872602,0.784091,0.876039,0.886364


In [ ]:
def paired_report(a, b, label):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    diff = b - a
    t_stat, p_t = stats.ttest_rel(b, a)
    try:
        w_stat, p_w = stats.wilcoxon(diff)
    except ValueError as exc:
        # n petit (<=3) : Wilcoxon peut être dégénéré (toutes les différences de même signe)
        w_stat, p_w = np.nan, np.nan
        print(f"[{label}] Wilcoxon non calculable ({exc})")
    print(f"--- {label} ---")
    print(f"  n paires = {len(a)}, delta moyen (retenue - sans coût) = {diff.mean():+.4f}")
    print(f"  t apparié : t({len(a) - 1}) = {t_stat:.2f}, p = {p_t:.3f}")
    if not np.isnan(p_w):
        print(f"  Wilcoxon signé : W = {w_stat:.1f}, p = {p_w:.3f}")
    return {
        "metric": label,
        "n": len(a),
        "mean_diff": diff.mean(),
        "t": t_stat,
        "p_t": p_t,
        "p_w": p_w,
    }


results = [
    paired_report(paired["val_f1_macro_sans_cout"], paired["val_f1_macro_retenue"], "val_f1_macro"),
    paired_report(
        paired["val_min_critical_recall_sans_cout"],
        paired["val_min_critical_recall_retenue"],
        "val_min_critical_recall",
    ),
]
pd.DataFrame(results)


--- val_f1_macro ---
  n paires = 3, delta moyen (retenue - sans coût) = +0.0038
  t apparié : t(2) = 0.34, p = 0.768
  Wilcoxon signé : W = 2.0, p = 0.750
--- val_min_critical_recall ---
  n paires = 3, delta moyen (retenue - sans coût) = +0.0650
  t apparié : t(2) = 3.48, p = 0.074
  Wilcoxon signé : W = 0.0, p = 0.250


,metric,n,mean_diff,t,p_t,p_w
0,val_f1_macro,3,0.003800,0.336842,0.768298,0.75
1,val_min_critical_recall,3,0.064966,3.481534,0.073519,0.25


**Lecture.** Avec seulement **3 paires** (une par seed), la puissance statistique est très
faible : le test de Wilcoxon apparié est même dégénéré (avec n=3, la plus petite p-value
atteignable est 0,25, donc jamais significative au seuil usuel) et le t apparié n'a que 2
degrés de liberté. Les deux tests sont donc à interpréter comme un **complément
descriptif** de la comparaison du tableau `tab:imbalance-grid`, pas comme une preuve
d'écart significatif : la conclusion du rapport (« cumuler les trois leviers aide » sur
`type`) repose sur la cohérence de la direction de l'effet (rappel critique minimal en
nette hausse sur les 3 seeds) plus que sur la significativité statistique, qu'un plan à
3 seeds ne peut pas trancher fermement.

## 3. Rééquilibrage — état des lieux côté `criticality`

Le rapport signale que côté `criticality`, la comparaison « sans coût » (class-balanced +
focale, retenue avant l'ajout du coût clinique) n'a qu'**un seul seed** dans le
leaderboard, contrairement à la configuration retenue (coût calibré seul) qui dispose
déjà des seeds 0/1/2. On vérifie ici précisément ce qui manque pour pouvoir apparier.

In [ ]:
grid_crit = df[
    (df["task"] == "criticality")
    & (df["name"].isin(["grid_cost_matrix_1", "grid_citicality_loss_1", "grid_citicality_loss_2"]))
].copy()

sans_cout_crit = select_config(
    grid_crit,
    **{
        "loss.beta": 0.9999,
        "loss.gamma": 2.0,
        "loss.lambda_cost": 0.0,
        "train.sampler": "none",
        "train.sampler_alpha": 1.0,
    },
)
retenue_crit = select_config(
    grid_crit,
    **{
        "loss.beta": 0.0,
        "loss.gamma": 0.0,
        "loss.lambda_cost": 1.0,
        "loss.cost_matrix": "[[0.0, 0.5, 1.0], [2.0, 0.0, 1.0], [4.0, 2.0, 0.0]]",
        "train.sampler": "none",
        "train.sampler_alpha": 1.0,
    },
)

seeds_sans_cout = sorted(sans_cout_crit["seed"].unique())
seeds_retenue = sorted(s for s in retenue_crit["seed"].unique() if s in (0, 1, 2))
missing = [s for s in (0, 1, 2) if s not in seeds_sans_cout]

print(
    "sans coût  — seeds disponibles :",
    seeds_sans_cout,
    "| val_f1_macro :",
    sans_cout_crit["val_f1_macro"].tolist(),
)
print("retenue    — seeds disponibles (0/1/2) :", seeds_retenue)
print()
if missing:
    print(
        f"Appariement impossible : il manque les seeds {missing} pour la configuration "
        "« sans coût » (class-balanced + focale, beta=0.9999, gamma=2). "
        f"À rejouer avant de pouvoir refaire ce test (cf. \\todo du rapport)."
    )
else:
    paired_crit = sans_cout_crit[["seed", "val_f1_macro", "val_min_critical_recall"]].merge(
        retenue_crit[["seed", "val_f1_macro", "val_min_critical_recall"]],
        on="seed",
        suffixes=("_sans_cout", "_retenue"),
    )
    paired_report(
        paired_crit["val_f1_macro_sans_cout"],
        paired_crit["val_f1_macro_retenue"],
        "val_f1_macro (criticality)",
    )
    paired_report(
        paired_crit["val_min_critical_recall_sans_cout"],
        paired_crit["val_min_critical_recall_retenue"],
        "val_min_critical_recall (criticality)",
    )


sans coût  — seeds disponibles : [np.int64(0)] | val_f1_macro : [0.9106007814407348, 0.9190727472305298]
retenue    — seeds disponibles (0/1/2) : [np.int64(0), np.int64(1), np.int64(2)]

Appariement impossible : il manque les seeds [1, 2] pour la configuration « sans coût » (class-balanced + focale, beta=0.9999, gamma=2). À rejouer avant de pouvoir refaire ce test (cf. \todo du rapport).


**Conclusion.** Confirmé : la configuration « sans coût » de `criticality` n'a qu'un run
au seed 0 dans le leaderboard actuel — le test apparié n'est pas calculable en l'état. Il
faut relancer cette configuration (`loss.beta=0,9999`, `loss.gamma=2`, `loss.lambda_cost=0`,
sans matrice de coût, sans sampler) sur les seeds **1** et **2** pour l'aligner sur la
configuration retenue (déjà disponible sur 0/1/2) avant de pouvoir reproduire ici le même
test apparié que pour `type`.